# AmsterdamUMCdb Concept table Exploration

## Objectives
- Search the concept table for ventilation, ECMO, and candidate feature terms.
- Validate each concept against the data by visit/row counts.
- Using the concept_ancestor hierarchy to find concepts a keyword search misses.
- Confirm which domain each concept lives in(Procedure, Measurement, etc.)
- Output a validated concept list for the cohort and feature notebooks.

### Imports

In [1]:
import pandas as pd
import numpy as np
from google.cloud import bigquery
import matplotlib.pyplot as plt

Initial BigQuery Setup by Ayushi Kashyap - Adapted for this notebook.

### Retrieving Google Project Id

In [2]:
client = bigquery.Client()

print("Connected to BigQuery")

Connected to BigQuery


In [3]:
# sets *your* project id
PROJECT_ID = "capstoneweaningprediction" #@param {type:"string"}

# Sets the default BigQuery dataset for accessing AmsterdamUMCdb

If you have received instructions to use a specific BigQuery instance, change the default settings here. Otherwise use these default values.

In [4]:
# sets default dataset for AmsterdamUMCdb
DATASET_PROJECT_ID = 'amsterdamumcdb' #@param {type:"string"}
DATASET_ID = 'version1_5_0' #@param {type:"string"}
LOCATION = 'eu' #@param {type:"string"}

# Provide your credentials to access the AmsterdamUMCdb dataset on Google BigQuery
Authenticate your credentials with Google Cloud Platform and set your default Google Cloud Project ID as an environment variable for running query jobs.

1. Run the cell. The `Allow this notebook to access your Google credentials?` prompt appears. Select `Allow`.
2. In the `Sign in - Google Accounts` dialog, use the account you registered during the AmsterdamUMCdb application process and select `Allow` again.

In [5]:
import os
from google.colab import auth

# all libraries check this environment variable, so set it:
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID

auth.authenticate_user()
print('Authenticated')

Authenticated


# Enable data table display

Colab includes the `google.colab.data_table` package that can be used to display Pandas dataframes as an interactive data table (default limits: `max_rows = 20000`, `max_columns = 20`). This is especially useful when exploring the  tables or dictionary from AmsterdamUMCdb. It can be enabled with:

In [6]:
%load_ext google.colab.data_table
from google.colab.data_table import DataTable

# change default limits:
DataTable.max_columns = 50
DataTable.max_rows = 80000


## Set the default query job configuration for magics

In [7]:
%load_ext bigquery_magics
from bigquery_magics import bigquery_magics
from google.cloud import bigquery

# sets the default query job configuration
def_config = bigquery.job.QueryJobConfig(default_dataset=DATASET_PROJECT_ID + "." + DATASET_ID)
bigquery_magics.context.default_query_job_config = def_config

## Query the `person` table and copy the data to the `persons` Pandas dataframe:

The `person` table contains a record for each patient in AmsterdamUMCdb.

Since this is a relatively small table, it is acceptable to use `SELECT *`.

**Note**: Should an error occur while running the query, please see
the AmsterdamUMCdb BigQuery [Frequently Asked Questions](https://github.com/AmsterdamUMC/AmsterdamUMCdb/wiki/bigquery#faq).

In [8]:
%%bigquery person
SELECT * FROM `amsterdamumcdb.version1_5_0.person`;

Query is running:   0%|          |

Downloading:   0%|          |


## Set the default query job configuration for google-cloud-bigquery client

In [9]:
from google.cloud import bigquery

# BigQuery requires a separate config to prevent the 'BadRequest: 400 Cannot explicitly modify anonymous table' error message
job_config = bigquery.job.QueryJobConfig()

# sets default client settings by re-using the previously defined config
client = bigquery.Client(project=PROJECT_ID, location=LOCATION, default_query_job_config=def_config)

### Explore Concept Table

In [10]:
query = f"""
SELECT *
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept`
"""

concept_df = client.query(query).to_dataframe()

concept_df.head()

,concept_id,concept_name,domain_id,vocabulary_id,concept_class_id,standard_concept,concept_code,valid_start_date,valid_end_date,invalid_reason
0,2617621,CLINICIAN DOCUMENTED THAT PROPHYLACTIC ANTIBIO...,Observation,HCPCS,HCPCS,S,G8211,1970-01-01,2011-10-30,None
1,2617574,Patient documented as being treated with antid...,Observation,HCPCS,HCPCS,S,G8129,1970-01-01,2014-11-11,None
2,2721144,UTERINE ARTERY EMBOLIZATION FOR UTERINE FIBROI...,Procedure,HCPCS,HCPCS,S,S2250,1970-01-01,2011-10-30,None
3,2720703,"ALS VEHICLE USED, NON-EMERGENCY TRANSPORT, NO ...",Observation,HCPCS,HCPCS,None,Q3020,1970-01-01,2011-01-09,None
4,2617676,Patient not documented to have received plan o...,Observation,HCPCS,HCPCS,S,G8268,1970-01-01,2014-11-11,None


#### Explore Domain Distribution

In [11]:
query = f"""
SELECT domain_id, COUNT(*) AS count
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept`
GROUP BY domain_id
ORDER BY count DESC
"""

concept_domain_df = client.query(query).to_dataframe()

concept_domain_df

,domain_id,count
0,Drug,4620617
1,Observation,378247
2,Condition,258368
3,Device,236567
4,Geography,204035
5,Measurement,186645
6,Procedure,99306
7,Spec Anatomic Site,40148
8,Meas Value,25187
9,Metadata,4759


Measurement, Device, and Procedure are well represented in the database.

#### Explore Vocabulary Distribution

In [12]:
query = f"""
SELECT vocabulary_id, COUNT(*) AS count
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept`
GROUP BY vocabulary_id
ORDER BY count DESC
"""

concept_vocabulary_df = client.query(query).to_dataframe()

concept_vocabulary_df

,vocabulary_id,count
0,RxNorm Extension,2145325
1,NDC,1195711
2,SNOMED,1079403
3,SPL,721549
4,RxNorm,308709
5,LOINC,269552
6,OSM,203339
7,ICD10CM,99130
8,ICD9CM,17564
9,HCPCS,11948


RxNorm Extension and RxNorm together dominate the vocabulary

#### Search Ventilation Concepts

In [13]:
query = f"""
SELECT concept_id, concept_name, domain_id
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept`
WHERE Lower(concept_name) LIKE '%ventilation%'
"""

ventilation_concept_df = client.query(query).to_dataframe()

ventilation_concept_df


,concept_id,concept_name,domain_id
0,43021811,Sleep related hypoventilation or hypoxemia,Condition
1,42530690,Invasive mechanical ventilation by intubation ...,Meas Value
2,42527120,Expired minute Volume during Mechanical ventil...,Measurement
3,42527121,Inspired minute Volume during Mechanical venti...,Measurement
4,42535241,Dependence on biphasic positive airway pressur...,Observation
...,...,...,...
475,4266619,Ventilation detail,Measurement
476,4229907,"Assisted ventilation therapy, pressure or volu...",Procedure
477,4283075,"Flow and timed ventilation procedure, initiati...",Procedure
478,4164571,Positive end expiratory pressure ventilation t...,Procedure


### Ventillation Concepts With Count

 AI-assisted (Claude, Anthropic — Claude Opus 4.8): ventilation-concept search with visit-count validation drafted with AI.

In [14]:
query = f"""
SELECT
    c.concept_id,
    c.concept_name,
    c.domain_id,
    COUNT(DISTINCT po.visit_occurrence_id) AS visit_count
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept` c
LEFT JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.procedure_occurrence` po
       ON po.procedure_concept_id = c.concept_id
      AND po.provider_id IS NOT NULL
WHERE LOWER(c.concept_name) LIKE '%ventilation%'
GROUP BY c.concept_id, c.concept_name, c.domain_id
ORDER BY visit_count DESC
"""
ventilation_concept_df = client.query(query).to_dataframe()
print(f"Concepts matching 'ventilation': {len(ventilation_concept_df)}")
print(f"With data in procedure_occurrence: {(ventilation_concept_df['visit_count'] > 0).sum()}")
ventilation_concept_df

Concepts matching 'ventilation': 480
With data in procedure_occurrence: 14


,concept_id,concept_name,domain_id,visit_count
0,37154096,Pressure support ventilation,Procedure,14528
1,37151337,Continuous mandatory ventilation pressure-cont...,Procedure,14276
2,37152413,Continuous mandatory ventilation volume-contro...,Procedure,6046
3,40486624,Non-invasive positive pressure ventilation,Procedure,2012
4,4165535,Continuous positive airway pressure ventilatio...,Procedure,915
...,...,...,...,...
475,1026399,Views ventilation and perfusion,Observation,0
476,1024805,Multisection ventilation+perfusion^W radionucl...,Observation,0
477,1024609,Voluntary ventilation.max^pre bronchodilator/V...,Observation,0
478,1024407,Breaths^at maximum voluntary ventilation,Observation,0


#### Search Extubation Concepts

In [15]:
query = f"""
SELECT concept_id, concept_name, domain_id
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept`
WHERE Lower(concept_name) LIKE '%extubation%'
"""

extubation_concept_df = client.query(query).to_dataframe()

extubation_concept_df

,concept_id,concept_name,domain_id
0,45882213,Unplanned extubation,Meas Value
1,3657545,Recently performed an extubation of trachea,Observation
2,3040451,Extubation urgency,Observation
3,36304219,Time of extubation,Observation
4,42537924,Accidental awareness under general anesthesia ...,Observation
5,1027261,Extubation urgency,Observation
6,37049773,Time of extubation,Observation
7,37055966,Time of extubation | Patient | Respiratory mea...,Observation
8,37399087,Post extubation acute respiratory failure requ...,Condition
9,40378383,Extubation (& [removal of endotracheal tube]),Procedure


Extubation Concepts Count

AI-assisted (Claude, Anthropic — Claude Opus 4.8): extubation-concept search with multi-table visit-count validation drafted with AI.

In [16]:
query = f"""
WITH ext AS (
    SELECT concept_id, concept_name, domain_id
    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept`
    WHERE LOWER(concept_name) LIKE '%extubation%'
)
SELECT
    e.concept_id,
    e.concept_name,
    e.domain_id,
    (SELECT COUNT(DISTINCT visit_occurrence_id)
       FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.procedure_occurrence`
      WHERE procedure_concept_id = e.concept_id AND provider_id IS NOT NULL)  AS proc_visits,
    (SELECT COUNT(DISTINCT visit_occurrence_id)
       FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.observation`
      WHERE observation_concept_id = e.concept_id AND provider_id IS NOT NULL) AS obs_visits,
    (SELECT COUNT(DISTINCT visit_occurrence_id)
       FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`
      WHERE measurement_concept_id = e.concept_id AND provider_id IS NOT NULL) AS meas_visits
FROM ext e
ORDER BY proc_visits DESC, obs_visits DESC, meas_visits DESC
"""
extubation_concept_df = client.query(query).to_dataframe()
extubation_concept_df['total_visits'] = extubation_concept_df[['proc_visits','obs_visits','meas_visits']].sum(axis=1)
print(f"Concepts matching 'extubation': {len(extubation_concept_df)}")
print(f"With any data: {(extubation_concept_df['total_visits'] > 0).sum()}")
extubation_concept_df

Concepts matching 'extubation': 15
With any data: 0


,concept_id,concept_name,domain_id,proc_visits,obs_visits,meas_visits,total_visits
0,42537924,Accidental awareness under general anesthesia ...,Observation,0,0,0,0
1,4231838,Inadvertent tracheal extubation,Observation,0,0,0,0
2,4337736,Extubation of bronchus,Procedure,0,0,0,0
3,1027261,Extubation urgency,Observation,0,0,0,0
4,4236712,Difficulty with tracheal extubation,Observation,0,0,0,0
5,4124035,Extubation of gastrointestinal tract,Procedure,0,0,0,0
6,37055966,Time of extubation | Patient | Respiratory mea...,Observation,0,0,0,0
7,4148972,Extubation of trachea,Procedure,0,0,0,0
8,36304219,Time of extubation,Observation,0,0,0,0
9,37399087,Post extubation acute respiratory failure requ...,Condition,0,0,0,0


#### Search Intubation Concepts

In [17]:
query = f"""
SELECT concept_id, concept_name, domain_id
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept`
WHERE concept_name LIKE '%intubation%'
"""

intubation_concept_df = client.query(query).to_dataframe()

intubation_concept_df

,concept_id,concept_name,domain_id
0,42530690,Invasive mechanical ventilation by intubation ...,Meas Value
1,42536231,Video intubation laryngoscope kit,Device
2,42536230,Video intubation laryngoscope handle/monitor,Device
3,42536219,Video intubation laryngoscope blade,Device
4,42537975,"Rigid intubation laryngoscope, single-use",Device
...,...,...,...
148,4175851,History of difficult intubation,Observation
149,4176465,History of intubation of gastrointestinal trac...,Observation
150,4134538,Unintended endobronchial intubation,Procedure
151,4160174,Inadvertent esophageal intubation,Procedure


### Intubation Visit Count

In [18]:
query = f"""
WITH intub AS (
    SELECT concept_id, concept_name, domain_id
    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept`
    WHERE LOWER(concept_name) LIKE '%intubation%'
)
SELECT
    i.concept_id,
    i.concept_name,
    i.domain_id,
    (SELECT COUNT(DISTINCT visit_occurrence_id)
       FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.procedure_occurrence`
      WHERE procedure_concept_id = i.concept_id AND provider_id IS NOT NULL)  AS proc_visits,
    (SELECT COUNT(DISTINCT visit_occurrence_id)
       FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.observation`
      WHERE observation_concept_id = i.concept_id AND provider_id IS NOT NULL) AS obs_visits,
    (SELECT COUNT(DISTINCT visit_occurrence_id)
       FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`
      WHERE measurement_concept_id = i.concept_id AND provider_id IS NOT NULL) AS meas_visits
FROM intub i
ORDER BY proc_visits DESC, obs_visits DESC, meas_visits DESC
"""
intubation_concept_df = client.query(query).to_dataframe()
intubation_concept_df['total_visits'] = intubation_concept_df[['proc_visits','obs_visits','meas_visits']].sum(axis=1)
print(f"Concepts matching 'intubation': {len(intubation_concept_df)}")
print(f"With any data: {(intubation_concept_df['total_visits'] > 0).sum()}")
intubation_concept_df

Concepts matching 'intubation': 276
With any data: 3


,concept_id,concept_name,domain_id,proc_visits,obs_visits,meas_visits,total_visits
0,4202832,Intubation,Procedure,1035,0,0,1035
1,3046004,Intubation tube depth Respiratory system,Measurement,0,0,9171,9171
2,3028878,Tube cuff pressure Intubation tube,Measurement,0,0,5566,5566
3,4123572,Intubation of gastrointestinal tract via colos...,Procedure,0,0,0,0
4,1761298,COVID-19 Intubation Progress note | Hospital |...,Note,0,0,0,0
...,...,...,...,...,...,...,...
271,37159176,At increased risk for difficult tracheal intub...,Observation,0,0,0,0
272,3544483,Intubation of rectum,Procedure,0,0,0,0
273,3566295,Digital assisted intubation,Procedure,0,0,0,0
274,4066650,Intubation of jejunum for measurement of intes...,Procedure,0,0,0,0


### Identify Respiratory Measurements

In [19]:
query = f"""
SELECT concept_id, concept_name, domain_id
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept`
WHERE lower(concept_name) LIKE '%respiratory%'
"""

respiratory_concept_df = client.query(query).to_dataframe()

respiratory_concept_df

,concept_id,concept_name,domain_id
0,46238011,Functions of the respiratory system,Meas Value
1,46237196,Medical-Respiratory Distress-Bronchiolitis,Meas Value
2,46237640,Medical-Respiratory Distress-Croup,Meas Value
3,46237772,"Structures of the cardiovascular, immunologica...",Meas Value
4,46366214,budesonide .25mg/2mL RESPIRATORY (INHALATION) ...,Drug
...,...,...,...
18068,36142172,oxygen 99L/100L RESPIRATORY (INHALATION) GAS,Drug
18069,36378163,oxygen 99L/100L RESPIRATORY (INHALATION) GAS,Drug
18070,35908222,oxygen 99L/100L RESPIRATORY (INHALATION) GAS,Drug
18071,36335412,oxygen 99L/100L RESPIRATORY (INHALATION) GAS,Drug


### Respiratory concepts visit count

In [20]:
query = f"""
WITH resp AS (
    SELECT concept_id, concept_name, domain_id
    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept`
    WHERE LOWER(concept_name) LIKE '%respiratory%'
)
SELECT
    r.concept_id,
    r.concept_name,
    r.domain_id,
    (SELECT COUNT(DISTINCT visit_occurrence_id)
       FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`
      WHERE measurement_concept_id = r.concept_id AND provider_id IS NOT NULL) AS meas_visits,
    (SELECT COUNT(DISTINCT visit_occurrence_id)
       FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.procedure_occurrence`
      WHERE procedure_concept_id = r.concept_id AND provider_id IS NOT NULL)  AS proc_visits,
    (SELECT COUNT(DISTINCT visit_occurrence_id)
       FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.observation`
      WHERE observation_concept_id = r.concept_id AND provider_id IS NOT NULL) AS obs_visits,
    (SELECT COUNT(DISTINCT visit_occurrence_id)
       FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.condition_occurrence`
      WHERE condition_concept_id = r.concept_id AND provider_id IS NOT NULL) AS cond_visits
FROM resp r
ORDER BY meas_visits DESC, proc_visits DESC, obs_visits DESC, cond_visits DESC
"""
respiratory_concept_df = client.query(query).to_dataframe()
respiratory_concept_df['total_visits'] = respiratory_concept_df[['meas_visits','proc_visits','obs_visits','cond_visits']].sum(axis=1)
print(f"Concepts matching 'respiratory': {len(respiratory_concept_df)}")
print(f"With any data: {(respiratory_concept_df['total_visits'] > 0).sum()}")
respiratory_concept_df.head(40)

Concepts matching 'respiratory': 18073
With any data: 48


,concept_id,concept_name,domain_id,meas_visits,proc_visits,obs_visits,cond_visits,total_visits
0,3024171,Respiratory rate,Measurement,21962,0,0,0,21962
1,3016078,Maximum [Pressure] Respiratory system airway o...,Measurement,16437,0,0,0,16437
2,21490855,PEEP Respiratory system --on ventilator,Measurement,16431,0,0,0,16431
3,3025408,Oxygen/Inspired gas Respiratory system by O2 A...,Measurement,15694,0,0,0,15694
4,3015016,Tidal volume expired spontaneous+mechanical Re...,Measurement,15527,0,0,0,15527
5,21490580,Carbon dioxide production (VCO2) in Respirator...,Measurement,13272,0,0,0,13272
6,3046004,Intubation tube depth Respiratory system,Measurement,9171,0,0,0,9171
7,21490737,Airway pressure --during respiratory pause,Measurement,6409,0,0,0,6409
8,42527140,Total PEEP Respiratory system,Measurement,1774,0,0,0,1774
9,21490789,Tidal volume expired Respiratory system airway...,Measurement,942,0,0,0,942


#### Explore Peep Concepts

In [21]:
query = f"""
SELECT concept_id, concept_name, domain_id
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept`
WHERE lower(concept_name) LIKE '%peep%'
"""

peep_concept_df = client.query(query).to_dataframe()

peep_concept_df

,concept_id,concept_name,domain_id
0,45771559,"PEEP valve, reusable",Device
1,45758300,"PEEP valve, single-use",Device
2,37025932,Intrinsic dynamic PEEP,Observation
3,37065391,Total dynamic PEEP,Observation
4,37028469,Extrinsic PEEP,Observation
5,37062926,Extrinsic dynamic PEEP,Observation
6,37076032,PEEP,Observation
7,37047538,Total PEEP,Observation
8,37057099,Intrinsic dynamic PEEP | Respiratory system | ...,Measurement
9,37030304,Total dynamic PEEP | Respiratory system | Resp...,Measurement


### Peep Visit Count

In [22]:
query = f"""
WITH peep AS (
    SELECT concept_id, concept_name, domain_id
    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept`
    WHERE LOWER(concept_name) LIKE '%peep%'
)
SELECT
    p.concept_id,
    p.concept_name,
    p.domain_id,
    (SELECT COUNT(DISTINCT visit_occurrence_id)
       FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`
      WHERE measurement_concept_id = p.concept_id AND provider_id IS NOT NULL)                           AS meas_visits,
    (SELECT COUNT(DISTINCT visit_occurrence_id)
       FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.device_exposure`
      WHERE device_concept_id = p.concept_id AND provider_id IS NOT NULL)     AS device_visits,
    (SELECT COUNT(DISTINCT visit_occurrence_id)
       FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.observation`
      WHERE observation_concept_id = p.concept_id AND provider_id IS NOT NULL)                           AS obs_visits
FROM peep p
ORDER BY meas_visits DESC, device_visits DESC
"""
peep_concept_df = client.query(query).to_dataframe()
peep_concept_df['total_visits'] = peep_concept_df[['meas_visits','device_visits','obs_visits']].sum(axis=1)
print(f"Concepts matching 'peep': {len(peep_concept_df)}")
print(f"With any data: {(peep_concept_df['total_visits'] > 0).sum()}")
peep_concept_df

Concepts matching 'peep': 33
With any data: 3


,concept_id,concept_name,domain_id,meas_visits,device_visits,obs_visits,total_visits
0,21490855,PEEP Respiratory system --on ventilator,Measurement,16431,0,0,16431
1,42527140,Total PEEP Respiratory system,Measurement,1774,0,0,1774
2,3035822,Intrinsic PEEP Respiratory system,Measurement,148,0,0,148
3,37022872,Extrinsic PEEP | Respiratory system | Respirat...,Measurement,0,0,0,0
4,45758300,"PEEP valve, single-use",Device,0,0,0,0
5,37076032,PEEP,Observation,0,0,0,0
6,37047691,Extrinsic dynamic PEEP | Respiratory system | ...,Measurement,0,0,0,0
7,42528516,Extrinsic dynamic PEEP Respiratory system,Measurement,0,0,0,0
8,37037082,Total PEEP | Respiratory system | Respiratory ...,Measurement,0,0,0,0
9,42528517,Total dynamic PEEP Respiratory system,Measurement,0,0,0,0


#### Explore Mechanical Ventilation Concepts

In [23]:
query = f"""
SELECT concept_id, concept_name, domain_id
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept`
WHERE lower(concept_name) LIKE '%mechanical ventilation%'
"""

mechanical_ventilation_concept_df = client.query(query).to_dataframe()

mechanical_ventilation_concept_df

,concept_id,concept_name,domain_id
0,37158404,Invasive mechanical ventilation,Procedure
1,42530690,Invasive mechanical ventilation by intubation ...,Meas Value
2,42527120,Expired minute Volume during Mechanical ventil...,Measurement
3,42527121,Inspired minute Volume during Mechanical venti...,Measurement
4,1035133,"Yes, on invasive mechanical ventilation suppor...",Meas Value
5,1620591,"Hospitalized - severe disease, intubation and ...",Meas Value
6,1017310,Invasive mechanical ventilation support upon a...,Observation
7,801179,"Injection, tocilizumab, for hospitalized adult...",Drug
8,801187,"Intravenous infusion, tocilizumab, for hospita...",Drug
9,801188,"Intravenous infusion, tocilizumab, for hospita...",Drug


### Mechanical Ventillation Visit Count

In [24]:
query = f"""
WITH mv AS (
    SELECT concept_id, concept_name, domain_id
    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept`
    WHERE LOWER(concept_name) LIKE '%mechanical ventilation%'
)
SELECT
    m.concept_id,
    m.concept_name,
    m.domain_id,
    (SELECT COUNT(DISTINCT visit_occurrence_id)
       FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.procedure_occurrence`
      WHERE procedure_concept_id = m.concept_id AND provider_id IS NOT NULL)  AS proc_visits,
    (SELECT COUNT(DISTINCT visit_occurrence_id)
       FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`
      WHERE measurement_concept_id = m.concept_id AND provider_id IS NOT NULL)                           AS meas_visits,
    (SELECT COUNT(DISTINCT visit_occurrence_id)
       FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.observation`
      WHERE observation_concept_id = m.concept_id AND provider_id IS NOT NULL)                           AS obs_visits
FROM mv m
ORDER BY proc_visits DESC, meas_visits DESC, obs_visits DESC
"""
mechanical_ventilation_concept_df = client.query(query).to_dataframe()
mechanical_ventilation_concept_df['total_visits'] = mechanical_ventilation_concept_df[['proc_visits','meas_visits','obs_visits']].sum(axis=1)
print(f"Concepts matching 'mechanical ventilation': {len(mechanical_ventilation_concept_df)}")
print(f"With any data: {(mechanical_ventilation_concept_df['total_visits'] > 0).sum()}")
mechanical_ventilation_concept_df

Concepts matching 'mechanical ventilation': 31
With any data: 2


,concept_id,concept_name,domain_id,proc_visits,meas_visits,obs_visits,total_visits
0,42527120,Expired minute Volume during Mechanical ventil...,Measurement,0,15538,0,15538
1,42527121,Inspired minute Volume during Mechanical venti...,Measurement,0,15512,0,15512
2,36308558,"No, not on invasive mechanical ventilation sup...",Meas Value,0,0,0,0
3,2614873,"Moisture exchanger, disposable, for use with i...",Device,0,0,0,0
4,1035133,"Yes, on invasive mechanical ventilation suppor...",Meas Value,0,0,0,0
5,37158404,Invasive mechanical ventilation,Procedure,0,0,0,0
6,40312603,Mechanical ventilation (& intermittent positiv...,Procedure,0,0,0,0
7,1017310,Invasive mechanical ventilation support upon a...,Observation,0,0,0,0
8,1620591,"Hospitalized - severe disease, intubation and ...",Meas Value,0,0,0,0
9,36034360,Whether the patient received mechanical ventil...,Observation,0,0,0,0


### Validate candidate feature concepts ([Zappalà et al. predictors ](https://www.sciencedirect.com/science/article/pii/S0883944125000929))

AI-assisted (Claude, Anthropic — Claude Opus 4.8): predictor-to-concept search with visit validation drafted with AI.

In [25]:
# Candidate feature concepts from Zappalà et al. predictor variables.
# Each term searched + visit-counted across the domains it could live in.

predictors = {
    'heart_rate':        ['%heart rate%'],
    'blood_pressure':    ['%arterial blood pressure%', '%mean blood pressure%'],
    'respiratory_rate':  ['%respiratory rate%'],
    'spo2':              ['%oxygen saturation%', '%spo2%'],
    'fio2':              ['%fio2%', '%inspired oxygen%'],
    'peep':              ['%peep%'],
    'tidal_volume':      ['%tidal volume%'],
    'plateau_pressure':  ['%plateau pressure%'],
    'pao2':              ['%pao2%', '%oxygen [partial pressure] in arterial%'],
    'paco2':             ['%paco2%', '%carbon dioxide [partial pressure] in arterial%'],
    'ph':                ['%ph of arterial blood%'],
    'lactate':           ['%lactate%'],
    'sedation_score':    ['%sedation%', '%richmond%', '%rass%'],
    'weight':            ['%body weight%'],
}

rows = []
for feat, terms in predictors.items():
    like = ' OR '.join([f"LOWER(c.concept_name) LIKE '{t}'" for t in terms])
    query = f"""
    SELECT c.concept_id, c.concept_name, c.domain_id,
           (SELECT COUNT(DISTINCT visit_occurrence_id)
              FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`
             WHERE measurement_concept_id = c.concept_id AND provider_id IS NOT NULL) AS meas_visits
    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept` c
    WHERE ({like}) AND c.standard_concept = 'S'
    ORDER BY meas_visits DESC
    LIMIT 3
    """
    df = client.query(query).to_dataframe()
    df.insert(0, 'feature', feat)
    rows.append(df)

candidate_features = pd.concat(rows, ignore_index=True)
print(candidate_features.to_string(index=False))

         feature  concept_id                                                                                                                                                                                                                                         concept_name   domain_id  meas_visits
      heart_rate    21490872                                                                                                                                                                                                                       Heart rate.beat-to-beat by EKG Measurement        23041
      heart_rate     3027018                                                                                                                                                                                                                                           Heart rate Measurement          447
      heart_rate     3019995                                                                           

heart rate, blood pressure, respiratory rate, SpO_2, PEEP, tidal volume, lactate, sedation score are well recorded and ready to use some concepts showing no data need to modify the search term.